<a href="https://colab.research.google.com/github/marensofiesteen/IND310/blob/main/Notebook%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install and prepare libraries

The "names" package in Python is install to generate syntetic names for patients

In [1]:
pip install names

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.1/789.1 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for names: filename=names-0.3.0-py3-none-any.whl size=803749 sha256=cef26bb934c30763fe3ee87589a2587cf4304475fee6e2b8ac1362aa1fe0c717
  Stored in directory: /root/.cache/pip/wheels/bd/d4/8f/03181acc8ebae4403c4a16423e9f68a0b0e65496d607d43462
Successfully built names


Furrther the Python libraries such as pandas, numpy, names, random, and datetime are imported

In [9]:
import pandas as pd
import numpy as np
import random  # Import the random module
import names   # Assuming you have the 'names' module installed
import uuid


We donwload a medical archive CSV file from a URL, with specific delimiter, and displays the first few rows of the resulting DataFrame containing medical diagnosis and symptom data.

In [3]:
Medical_archieve=pd.read_csv("https://pvexpect.com/Healthcare/Medical_archieve.csv", delimiter=";")
Medical_archieve.head(1)

,Diagnosis,Symptom,Examination,Examination_2,Behandlding,Avdeling,Common/uncommon,"Sannsynlighet for kritisk utfall, når symptom er påvist",Kritiskhet kategori,Interview reveal,...,Diagnoses,Diagnoses_temp,Required Room(s),Departement(s),Required Equipment(s),Legehåndbok,Virtuell sykehus,Kostnader($),Sykdoms beskrivelse (Engelsk),Sykdoms beskrivelse (Norsk)
0,Acute Rheumatic Fever,Rheumatic Fever,an echocardiogram,NaN,antibiotics,Cardiology,Uncommon,100 %,Medium Hazard,False,...,"Bacterial tonsillitis, Laryngitis, Bronchitis,...",0,"Any office, Diagnostic unit",Any,0.0,https://nhi.no/sykdommer/hjertekar/ulike-sykdo...,0,1100.0,Acute rheumatic fever is an inflammatory disea...,Akutt revmatisk feber er en inflammatorisk syk...


In [4]:
# Assuming you have already loaded the dataset into Medical_archieve
column_names = Medical_archieve.columns

# Print the column names
print(column_names)

Index(['Diagnosis', 'Symptom', 'Examination', 'Examination_2', 'Behandlding',
       'Avdeling', 'Common/uncommon',
       'Sannsynlighet for kritisk utfall, når symptom er påvist',
       'Kritiskhet kategori', 'Interview reveal', 'Symptom.1', 'ICON',
       'IsMainSymptom', 'DiscomfortLevel', 'PatientComplains', 'Hazard',
       'PatientMobility', 'Examination 1', 'Examination 2', 'Examination 3',
       'Treatment', 'ShameLevel', 'WalkAnim', 'WalkSpeedModifier',
       'CollapseSymptom', 'RiskOfCollapseStartHours', 'RiskOfCollapseEndHours',
       'CannotTalk', 'RiskOfDeathStartHours', 'RiskOfDeathEndHours',
       'CollapseProcedure', 'PlanAllTreatments', 'BleedingLevel',
       'Behandling (Engelsk)', 'Behandling (Norsk)',
       'Behandling beskrivelse (Engelsk)', 'Behandling beskrivelse (Norsk)',
       'Krav', 'Surgery room', 'Diagnoses', 'Diagnoses_temp',
       'Required Room(s)', 'Departement(s)', 'Required Equipment(s)',
       'Legehåndbok', 'Virtuell sykehus', 'Kostnader(

In [5]:
# Assuming you have already loaded the dataset into Medical_archieve
num_columns = Medical_archieve.shape[1]

# Print the number of columns
print("Number of columns:", num_columns)

Number of columns: 49


Then we develop a function, `generate_patient()`, that creates a random patient profile by selecting a disease, generating various patient attributes such as name, planned appointment, hazard level, timeslot, arrival time, revealed and unrevealed symptoms, and diagnosis based on a medical archive dataset.

Her har vi valgt features som er relevante til hospital-sim i notebook 2

In [10]:
def generate_patient():
    # Bruk symptomer som kan avdekkes i intervjuet
    tilgjengelige = Medical_archieve[
        (Medical_archieve["Interview reveal"] == True)
        & Medical_archieve["Diagnosis"].notna()
        & Medical_archieve["Symptom"].notna()
    ]

    # Velg en diagnose og opptil tre tilhørende symptomer
    diagnose = random.choice(
        tilgjengelige["Diagnosis"].unique().tolist()
    )

    symptomrader = tilgjengelige[
        tilgjengelige["Diagnosis"] == diagnose
    ].drop_duplicates(subset="Symptom")

    valgte = symptomrader.sample(n=min(3, len(symptomrader)))

    symptomer = valgte["Symptom"].astype(str).str.strip().tolist()

    # Bruk et hovedsymptom som hovedplage hvis det finnes
    hovedsymptomer = valgte[
        valgte["IsMainSymptom"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("TRUE")
    ]

    hovedplage = (
        str(hovedsymptomer.iloc[0]["Symptom"]).strip()
        if not hovedsymptomer.empty
        else symptomer[0]
    )

    # Syntetisk alder og kjønn til simuleringen
    kjonn = random.choice(["Male", "Female"])
    alder = random.randint(18, 90)
    puls = 80 #bpm
    blodtrykk = {"systolic": 120, "diastolic": 80}
    temperatur = 98.6 #Fahrenheit

    # Gjør symptomnavn uavhengige av store/små bokstaver
    symptomnavn = {symptom.strip().lower() for symptom in symptomer}

    if "high fever" in symptomnavn:
        temperatur = 103.10
    elif "fever" in symptomnavn:
        temperatur = 101.3
    if "tachycardia" in symptomnavn:
        puls = 110
    if "bradycardia" in symptomnavn:
        puls = 40
    if "hypertension" in symptomnavn:
        blodtrykk = {"systolic": 140, "diastolic": 90}
    if "hypotension" in symptomnavn:
        blodtrykk = {"systolic": 90, "diastolic": 60}
    if diagnose.strip().lower() == "stroke":
        blodtrykk = {"systolic": 160, "diastolic": 90}
    if diagnose.strip().lower() == "mild frostbite on hands":
        puls = 100
        temperatur = 95
    if diagnose.strip().lower() == "mild frostbite on legs":
        puls = 100
        temperatur = 95
    if diagnose.strip().lower() == "severe frostbite on hands":
        puls = 40
        temperatur = 89.6
    if diagnose.strip().lower() == "severe frostbite on legs":
        puls = 40
        temperatur = 89.6

    pasient_id = str(uuid.uuid4())

    profil = {
        "profile_id": pasient_id,
        "name": names.get_full_name(gender=kjonn.lower()),
        "age": alder,
        "gender": kjonn,
        "chief_complaint": hovedplage,
        "symptoms": symptomer,
        "medical_history": [],
        "current_medications": [],
        "allergies": [],
        "vital_signs": {
            "heart_rate": puls,
            "blood_pressure": blodtrykk,
            "temperature": temperatur,
        },
    }

    fasit = {
        "profile_id": pasient_id,
        "name": profil["name"],
        "reference_diagnosis": diagnose,
    }

    return profil, fasit

    """return {
        "name": names.get_full_name(gender=kjonn.lower()),
        "age": alder,
        "gender": kjonn,
        "chief_complaint": hovedplage,
        "symptoms": symptomer,
        "medical_history": [],
        "current_medications": [],
        "allergies": [],
        "vital_signs": {
            "heart_rate": puls,
            "blood_pressure": blodtrykk,
            "temperature": temperatur,
        },
    }"""

Lager en tabell med pasienter. Antallet kan justeres.

In [13]:
# Velg antall pasienter
#antall_pasienter = 3

# Generer pasientene
#pasienter = [generate_patient() for _ in range(antall_pasienter)]

# Samle dem i en tabell
#pasienter_df = pd.DataFrame(pasienter)

# Vis de første 10 pasientene
#pasienter_df.head(10)

antall_pasienter = 3

pasienter = []
fasitliste = []

for _ in range(antall_pasienter):
    profil, fasit = generate_patient()
    pasienter.append(profil)
    fasitliste.append(fasit)

display(pd.DataFrame(pasienter))
display(pd.DataFrame(fasitliste))

,profile_id,name,age,gender,chief_complaint,symptoms,medical_history,current_medications,allergies,vital_signs
0,ef8b0963-98a0-463f-b872-fa2af2c05667,Deborah Aita,87,Female,Swelling Knee,"[Swelling Knee, Limited Leg Motion, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
1,f12afa0e-8847-4b6d-9734-b9984f0770c4,Jessica Stamey,33,Female,Red Eye,"[Red Eye, Watery Eyes, Pain]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
2,ab17de22-eca7-41da-9865-3211411324ce,Wanda Rabinowitz,30,Female,Vomiting,"[Vomiting, Breathing Problems, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."


,profile_id,name,reference_diagnosis
0,ef8b0963-98a0-463f-b872-fa2af2c05667,Deborah Aita,Knee Bursitis
1,f12afa0e-8847-4b6d-9734-b9984f0770c4,Jessica Stamey,Cluster Headache
2,ab17de22-eca7-41da-9865-3211411324ce,Wanda Rabinowitz,Septicemic Plague


Laster ned pasientene som en fil, som må legges inn i notebook 2

In [16]:
import json
from google.colab import files

with open("genererte_pasienter.json", "w", encoding="utf-8") as fil:
    json.dump(pasienter, fil, ensure_ascii=False, indent=2, allow_nan=False)

with open("diagnosefasit.json", "w", encoding="utf-8") as fil:
    json.dump(fasitliste, fil, ensure_ascii=False, indent=2, allow_nan=False)

files.download("genererte_pasienter.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download("diagnosefasit.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>